# Thirty-day readmission evaluation
Generated assurance-corpus case. Review brief.md before interpreting results.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
data = pd.read_csv("dataset/encounters.csv", parse_dates=[
    "prediction_time", "followup_recorded_at"
])
TARGET = "readmitted_30d"
CUTOFF = pd.Timestamp("2023-01-01")

In [ ]:
train = data.loc[data["prediction_time"] < CUTOFF].copy()
holdout = data.loc[data["prediction_time"] >= CUTOFF].copy()
group_overlap = sorted(
    set(train["patient_id"]) & set(holdout["patient_id"])
)

In [ ]:
FEATURE_COLUMNS = ['age', 'prior_admits_180d', 'comorbidity_score', 'length_of_stay_days', 'discharge_acuity', 'last_hemoglobin']
NUMERIC_FEATURES = ['age', 'prior_admits_180d', 'comorbidity_score', 'length_of_stay_days', 'last_hemoglobin']
CATEGORICAL_FEATURES = ['discharge_acuity']

In [ ]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
transformers = [("numeric", numeric_pipeline, NUMERIC_FEATURES)]
if CATEGORICAL_FEATURES:
    categorical_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ])
    transformers.append(("categorical", categorical_pipeline, CATEGORICAL_FEATURES))

model = Pipeline([
    ("preprocess", ColumnTransformer(transformers)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=4107)),
])
model.fit(train[FEATURE_COLUMNS], train[TARGET])
probability = model.predict_proba(holdout[FEATURE_COLUMNS])[:, 1]
roc_auc = float(roc_auc_score(holdout[TARGET], probability))

evaluation = {
    "metric": "roc_auc",
    "value": roc_auc,
    "features": FEATURE_COLUMNS,
    "cutoff": CUTOFF.isoformat(),
    "train_rows": int(len(train)),
    "holdout_rows": int(len(holdout)),
    "train_patients": int(train["patient_id"].nunique()),
    "holdout_patients": int(holdout["patient_id"].nunique()),
    "group_overlap_count": len(group_overlap),
}
Path("evaluation.json").write_text(
    json.dumps(evaluation, indent=2, sort_keys=True),
    encoding="utf-8",
)
print(json.dumps(evaluation, indent=2, sort_keys=True))